In [3]:
import os
import numpy as np
import warnings 

from datetime import datetime

from ObservationDataUtil import ( 
     ObsDataReader,
    
)
from ModelDataUtil import (
    ModelDataReader,
    EnsembleMetricEvaluator
)


In [2]:
if __name__ == "__main__":
    #top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    #out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    #fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"

    top_path = "/global/cfs/cdirs/m4849/zhan391/e3sm_dart"
    out_path = "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/diag_dart"
    fig_path = "./figures"
    
    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    regnam = "global"
    plot_type = "model_vs_obs"
    period = "201112-201112" 
    frequency = "monthly"
    derive_monthly = True 
    
    # === Variable dictionary ===
    var_dict = {
        'FLUT':   {'alias' : 'FLUT',   'unit': 'W m$^{-2}$',    'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "CERES-OAFlux"},
        'PRECT':  {'alias' : 'PRECT',  'unit': 'mm day$^{-1}$', 'level': np.linspace(-2, 2, 51),   'comp': 'atm', 'ref': "GPCP"},
        'TMQ':    {'alias' : 'TMQ',    'unit': 'kg m$^{-2}$',   'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "ERA5"},
    }
    
    model_labels = {
        'CTRLEN10': "CTRL (ENS=10)",
        'DARTEN10': "EAM-DART (ENS=10)",
        'DARTEN20': "EAM-DART (ENS=20)",
        'DARTEN40': "EAM-DART (ENS=40)",
        'DARTEN80': "EAM-DART (ENS=80)",
    }

    model_list = ['CTRLEN10', 'DARTEN20', 'DARTEN40']

    # Ensure output path exists
    os.makedirs(out_path, exist_ok=True)

    # Optional: save run configuration for reproducibility
    config_file = os.path.join(out_path, "diagnostic_run_config.txt")
    with open(config_file, "w") as f:
        f.write(f"Run Time: {datetime.now().isoformat()}\n")
        f.write(f"Variables: {list(var_dict.keys())}\n")
        f.write(f"Models: {model_list}\n")
        f.write(f"Region: {regnam}, Period: {period}, Frequency: {frequency}\n")
        f.write(f"Experiment Base: {exp_base}\n")

    # === Loop over variables ===
    for var, meta in var_dict.items():
        try:
            print(f"[{datetime.now().isoformat()}] Processing {var} using {meta['ref']}")

            component = meta.get('comp', 'atm')
            ref_var = meta.get('alias', var)
            ref_dataset = meta.get('ref', 'ERA5')

            # Load reference data
            try:
                obs_reader =  ObsDataReader(
                    obsname=ref_dataset,
                    period=period,
                    frequency=frequency
                )
                obs = obs_reader.read(var=ref_var)
            except FileNotFoundError:
                print(f"[WARN] Skipping {var} — observation not found for {ref_var} ({ref_dataset})")
                continue
                
            # Load model ensemble data
            model_reader = ModelDataReader(
                base_path=top_path,
                exp_base=exp_base,
                regnam=regnam,
                frequency=frequency,
                period=period,
                component=component,
                derive_monthly=derive_monthly
            )
            data_dict = model_reader.read_variable_across_experiments(var, model_list)
            #print(data_dict)
            #print(obs.min(),obs.max())
            #print(xxxx)
            
            # Evaluate metrics and save
            evaluator = EnsembleMetricEvaluator(
                obs=obs,
                model_dict=data_dict,
                component=component,
                ref_dataset=ref_dataset,
                var=var,
                vunit=meta['unit'],
                output_path=out_path
            )

        except Exception as e:
            print(f"[ERROR] Failed to process {var}: {e}")


[2025-06-09T23:33:21.119947] Processing FLUT using CERES-OAFlux
Parsed time range: 2011-12-01 00:00:00 to 2011-12-01 23:59:59
2011


/global/homes/z/zhan391/.local/lib/python3.10/site-packages/xarray/conventions.py:286: SerializationWarning: variable 'CLDTOT' has multiple fill values {-999.0, -9999.0} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/global/homes/z/zhan391/.local/lib/python3.10/site-packages/xarray/conventions.py:286: SerializationWarning: variable 'CRE' has multiple fill values {-999.0, -9999.0} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/global/homes/z/zhan391/.local/lib/python3.10/site-packages/xarray/conventions.py:286: SerializationWarning: variable 'CRE_SRF' has multiple fill values {-999.0, -9999.0} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/global/homes/z/zhan391/.local/lib/python3.10/site-packages/xarray/conventions.py:286: SerializationWarning: variable 'EVAP' has multiple fill values {-9999.0, 1e+20} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/global/homes/z/zhan391/.loca

Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
Parsed time range: 2011-12-01 00:00:00 to 2011-12-01 23:59:59
read CTRLEN10 data from /pscratch/sd/z/zhan391/e3sm_dart/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read CTRLEN10 data from /pscratch/sd/z/zhan391/e3sm_dart/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read CTRLEN10 data from /pscratch/sd/z/zhan391/e3sm_dart/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read CTRLEN10 data from /pscratch/sd/z/zhan391/e3sm_dart/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read CTRLEN10 data from /pscratc

Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read DARTEN40 data from /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read DARTEN40 data from /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read DARTEN40 data from /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read DARTEN40 data from /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read DARTEN40 data from /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne

Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read CTRLEN10 data from /compyfs/zhan391/v3_dart_cda_scratch/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read CTRLEN10 data from /compyfs/zhan391/v3_dart_cda_scratch/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read CTRLEN10 data from /compyfs/zhan391/v3_dart_cda_scratch/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read CTRLEN10 data from /compyfs/zhan391/v3_dart_cda_scratch/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
Parsed time range: 2011-12-01 00:00:00 to 2011-12-01 23:59:59
read DARTEN20 da

Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read DARTEN40 data from /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read DARTEN40 data from /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read DARTEN40 data from /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read DARTEN40 data from /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily
Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
read DARTEN40 data from /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne

In [ ]:
if __name__ == "__main__":
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"

    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    regnam = "global"
    plot_type = "model_vs_obs"
    period = "201112-201112"  # monthly format
    frequency = "monthly"
    derive_monthly = True 
    
    # === Variable dictionary ===
    var_dict = {
        'TS':     {'alias': 'TS',      'unit': '$^o$C',         'level': np.linspace(-8, 8, 51),   'comp': 'atm', 'ref': "ERA5"},
        'TREFHT': {'alias' : 'TREFHT', 'unit': '$^o$C',         'level': np.linspace(-8, 8, 51),   'comp': 'atm', 'ref': "ERA5"},
        'PSL':    {'alias' : 'PSL',    'unit': 'hPa',           'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "ERA5"},
    }
    
    model_labels = {
        'CTRLEN10': "CTRL (ENS=10)",
        'DARTEN10': "EAM-DART (ENS=10)",
        'DARTEN20': "EAM-DART (ENS=20)",
        'DARTEN40': "EAM-DART (ENS=40)",
        'DARTEN80': "EAM-DART (ENS=80)",
    }

    model_list = ['CTRLEN10', 'DARTEN20', 'DARTEN40']

    # Ensure output path exists
    os.makedirs(out_path, exist_ok=True)

    # Optional: save run configuration for reproducibility
    config_file = os.path.join(out_path, "diagnostic_run_config.txt")
    with open(config_file, "w") as f:
        f.write(f"Run Time: {datetime.now().isoformat()}\n")
        f.write(f"Variables: {list(var_dict.keys())}\n")
        f.write(f"Models: {model_list}\n")
        f.write(f"Region: {regnam}, Period: {period}, Frequency: {frequency}\n")
        f.write(f"Experiment Base: {exp_base}\n")

    # === Loop over variables ===
    for var, meta in var_dict.items():
        try:
            print(f"[{datetime.now().isoformat()}] Processing {var} using {meta['ref']}")

            component = meta.get('comp', 'atm')
            ref_var = meta.get('alias', var)
            ref_dataset = meta.get('ref', 'ERA5')

            # Load reference data
            try:
                obs = ObsDataReader().read(
                    var=ref_var,
                    obsname=ref_dataset,
                    time=period,
                    frequency=frequency
                )
            except FileNotFoundError:
                print(f"[WARN] Skipping {var} — observation not found for {ref_var} ({ref_dataset})")
                continue

            # Load model ensemble data
            model_reader = ModelDataReader(
                base_path=top_path,
                exp_base=exp_base,
                regnam=regnam,
                frequency=frequency,
                period=period,
                component=component,
                derive_monthly=derive_monthly
            )
            data_dict = model_reader.read_variable_across_experiments(var, model_list)
            #print(data_dict)
            #print(obs)

            # Evaluate metrics and save
            evaluator = EnsembleMetricEvaluator(
                obs=obs,
                model_dict=data_dict,
                component=component,
                ref_dataset=ref_dataset,
                var=var,
                vunit=meta['unit'],
                output_path=out_path
            )

        except Exception as e:
            print(f"[ERROR] Failed to process {var}: {e}")

In [ ]:
if __name__ == "__main__":
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"

    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    regnam = "global"
    plot_type = "model_vs_obs"
    period = "201112-201112"  # monthly format
    frequency = "monthly"

    # === Variable dictionary ===
    var_dict = {
        'TS':     {'alias': 'TS',      'unit': '$^o$C',         'level': np.linspace(-8, 8, 51),   'comp': 'atm', 'ref': "ERA5"},
        'TREFHT': {'alias' : 'TREFHT', 'unit': '$^o$C',         'level': np.linspace(-8, 8, 51),   'comp': 'atm', 'ref': "ERA5"},
        'PSL':    {'alias' : 'PSL',    'unit': 'hPa',           'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "ERA5"},
    }
    
    model_labels = {
        'CTRLEN10': "CTRL (ENS=10)",
        'DARTEN10': "EAM-DART (ENS=10)",
        'DARTEN20': "EAM-DART (ENS=20)",
        'DARTEN40': "EAM-DART (ENS=40)",
        'DARTEN80': "EAM-DART (ENS=80)",
    }

    model_list = ['CTRLEN10', 'DARTEN20', 'DARTEN40']

    # Ensure output path exists
    os.makedirs(out_path, exist_ok=True)

    # Optional: save run configuration for reproducibility
    config_file = os.path.join(out_path, "diagnostic_run_config.txt")
    with open(config_file, "w") as f:
        f.write(f"Run Time: {datetime.now().isoformat()}\n")
        f.write(f"Variables: {list(var_dict.keys())}\n")
        f.write(f"Models: {model_list}\n")
        f.write(f"Region: {regnam}, Period: {period}, Frequency: {frequency}\n")
        f.write(f"Experiment Base: {exp_base}\n")

    # === Loop over variables ===
    for var, meta in var_dict.items():
        try:
            print(f"[{datetime.now().isoformat()}] Processing {var} using {meta['ref']}")

            component = meta.get('comp', 'atm')
            ref_var = meta.get('alias', var)
            ref_dataset = meta.get('ref', 'ERA5')

            # Load reference data
            try:
                obs = ObsDataReader().read(
                    var=ref_var,
                    obsname=ref_dataset,
                    time=period,
                    frequency=frequency
                )
            except FileNotFoundError:
                print(f"[WARN] Skipping {var} — observation not found for {ref_var} ({ref_dataset})")
                continue

            # Load model ensemble data
            model_reader = ModelDataReader(
                base_path=top_path,
                exp_base=exp_base,
                regnam=regnam,
                frequency=frequency,
                period=period,
                component=component
            )
            data_dict = model_reader.read_variable_across_experiments(var, model_list)
            #print(data_dict)
            #print(obs)

            # Evaluate metrics and save
            evaluator = EnsembleMetricEvaluator(
                obs=obs,
                model_dict=data_dict,
                component=component,
                ref_dataset=ref_dataset,
                var=var,
                vunit=meta['unit'],
                output_path=out_path
            )

        except Exception as e:
            print(f"[ERROR] Failed to process {var}: {e}")


In [6]:
if __name__ == "__main__":
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"

    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    regnam = "global"
    plot_type = "model_vs_obs"
    period = "201112-201112"  # monthly format
    frequency = "monthly"

    # === Variable dictionary ===
    var_dict = {
        'TS':     {'alias': 'TS',      'unit': '$^o$C',         'level': np.linspace(-8, 8, 51),   'comp': 'atm', 'ref': "ERA5"},
        'TREFHT': {'alias' : 'TREFHT', 'unit': '$^o$C',         'level': np.linspace(-8, 8, 51),   'comp': 'atm', 'ref': "ERA5"},
        'PSL':    {'alias' : 'PSL',    'unit': 'hPa',           'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "ERA5"},
    }
    
    model_labels = {
        'CTRLEN10': "CTRL (ENS=10)",
        'DARTEN10': "EAM-DART (ENS=10)",
        'DARTEN20': "EAM-DART (ENS=20)",
        'DARTEN40': "EAM-DART (ENS=40)",
        'DARTEN80': "EAM-DART (ENS=80)",
    }

    model_list = ['CTRLEN10', 'DARTEN20', 'DARTEN40']

    # Ensure output path exists
    os.makedirs(out_path, exist_ok=True)

    # Optional: save run configuration for reproducibility
    config_file = os.path.join(out_path, "diagnostic_run_config.txt")
    with open(config_file, "w") as f:
        f.write(f"Run Time: {datetime.now().isoformat()}\n")
        f.write(f"Variables: {list(var_dict.keys())}\n")
        f.write(f"Models: {model_list}\n")
        f.write(f"Region: {regnam}, Period: {period}, Frequency: {frequency}\n")
        f.write(f"Experiment Base: {exp_base}\n")

    # === Loop over variables ===
    for var, meta in var_dict.items():
        try:
            print(f"[{datetime.now().isoformat()}] Processing {var} using {meta['ref']}")

            component = meta.get('comp', 'atm')
            ref_var = meta.get('alias', var)
            ref_dataset = meta.get('ref', 'ERA5')

            # Load reference data
            try:
                obs = ObsDataReader().read(
                    var=ref_var,
                    obsname=ref_dataset,
                    time=period,
                    frequency=frequency
                )
            except FileNotFoundError:
                print(f"[WARN] Skipping {var} — observation not found for {ref_var} ({ref_dataset})")
                continue

            # Load model ensemble data
            model_reader = ModelDataReader(
                base_path=top_path,
                exp_base=exp_base,
                regnam=regnam,
                frequency=frequency,
                period=period,
                component=component
            )
            data_dict = model_reader.read_variable_across_experiments(var, model_list)
            #print(data_dict)
            #print(obs)

            # Evaluate metrics and save
            evaluator = EnsembleMetricEvaluator(
                obs=obs,
                model_dict=data_dict,
                component=component,
                ref_dataset=ref_dataset,
                var=var,
                vunit=meta['unit'],
                output_path=out_path
            )

        except Exception as e:
            print(f"[ERROR] Failed to process {var}: {e}")


[2025-06-08T15:56:15.304670] Processing TS using ERA5
[ERROR] Failed to process TS: Given date string "12" not likely a datetime, at position 0
[2025-06-08T15:56:15.306344] Processing TREFHT using ERA5
[ERROR] Failed to process TREFHT: Given date string "12" not likely a datetime, at position 0
[2025-06-08T15:56:15.307664] Processing PSL using ERA5
[ERROR] Failed to process PSL: Given date string "12" not likely a datetime, at position 0
